# HyPE（Hypothetical Prompt Embeddings）：把“可能的提问方式”提前索引到向量库

本节目标：实现 HyPE 的最小版本，并理解它与 HyDE 的关键差异。

## 核心直觉

- **HyDE**：在“查询时”把用户问题扩写成 hypothetical document，再去检索（有查询时开销）。
- **HyPE**：在“建库时”对每个 chunk 预先生成多条“可能的用户问题/提问方式”（hypothetical prompts/questions），把这些问题的向量都索引起来。

这样检索时就变成：

- 用户 query  ↔  预生成的问题（question-question matching）

由于这些问题的写法更接近真实用户提问，通常比“query ↔ chunk正文”更好对齐。

> 重要提醒：HyPE 的成本主要发生在**索引阶段**（每个 chunk 都要 LLM 生成问题），但查询阶段几乎不增加额外开销。

## 0) 导入依赖并加载环境变量

延续前面 notebook 的约定：从 `../.env` 读取 `OPENAI_API_KEY`。

In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path
from typing import Iterable, List, Tuple

import faiss
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv

load_dotenv("../.env")

# 让 `all_rag_techniques/` 子目录里的 notebook
# 也能导入上一级目录的 `helper_functions.py` / `evaluation/`
sys.path.insert(0, str(Path("..").resolve()))

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from helper_functions import retrieve_context_per_question, show_context
from evaluation.evalute_rag import evaluate_rag

/tmp/ipykernel_3243784/363582771.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


## 1) 常量配置

- `PATH`：待索引 PDF
- `LANGUAGE_MODEL_NAME`：用于生成“假设问题”的模型
- `EMBEDDING_MODEL_NAME`：用于 embedding 的模型
- `CHUNK_SIZE/CHUNK_OVERLAP`：切 chunk 参数（本节与 `simple_rag_cn.ipynb` 保持一致节奏）

In [2]:
PATH = Path("../data/Understanding_Climate_Change.pdf")
assert PATH.exists(), f"找不到 PDF: {PATH.resolve()}"

LANGUAGE_MODEL_NAME = "gpt-4o-mini"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

PATH

PosixPath('../data/Understanding_Climate_Change.pdf')

## 2) 为每个 chunk 生成“假设问题”（HyPE 的核心）

对每个 chunk，我们让 LLM 生成多条“如果用户想要用这段信息回答问题，他可能会怎么问？”

这些问题会被 embedding 并写入向量库，但**映射回的内容仍然是原始 chunk**（所以检索结果依然是 chunk）。

> 解析重点：HyPE 把“扩写/改写”从查询时挪到了索引时。

In [3]:
def _clean_lines(lines: Iterable[str]) -> List[str]:
    out: List[str] = []
    for raw in lines:
        s = raw.strip()
        if not s:
            continue
        # 去掉常见列表前缀：- / * / • / 1. / 1)
        s = re.sub(r"^\s*[\-\*•]\s+", "", s)
        s = re.sub(r"^\s*\d+\s*[\)\.]\s*", "", s)
        s = s.strip()
        if s:
            out.append(s)
    return out


prompt_questions = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个用于 HyPE 的问题生成器。\n"
            "给定一段文本（chunk），请生成若干条‘关键问题’，这些问题的答案应该能覆盖文本的主要信息点。\n"
            "要求：\n"
            "- 每行一个问题\n"
            "- 不要编号，不要前缀\n"
            "- 不要输出解释"
        ),
        ("user", "文本：\n{chunk_text}\n\n问题："),
    ]
)


def generate_hypothetical_questions(chunk_text: str) -> List[str]:
    llm = ChatOpenAI(model=LANGUAGE_MODEL_NAME, temperature=0)
    raw = (prompt_questions | llm).invoke({"chunk_text": chunk_text}).content
    # 部分模型喜欢用空行分隔；合并空行以稳定解析
    raw = raw.replace("\n\n", "\n")
    return _clean_lines(raw.splitlines())


def generate_hypothetical_prompt_embeddings(chunk_text: str) -> Tuple[str, List[List[float]]]:
    questions = generate_hypothetical_questions(chunk_text)
    embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)
    vectors = embedding_model.embed_documents(questions)
    return chunk_text, vectors

## 3) 用“问题 embedding”填充 FAISS（一个 chunk 对应多条向量）

这里是 HyPE 和普通 RAG 的最大结构差异：

- 普通 RAG：每个 chunk → 1 个向量
- HyPE：每个 chunk → N 个向量（N = 生成出来的问题数量）

但重要的是：**这 N 个向量都指向同一个 chunk 内容**（因此检索返回的仍然是 chunk）。

In [4]:
def _replace_tabs(text: str) -> str:
    return text.replace("\t", " ")


def prepare_vector_store(chunks: List[str], *, max_workers: int | None = None) -> FAISS:
    # 延迟初始化：先拿到向量维度
    vector_store: FAISS | None = None

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(generate_hypothetical_prompt_embeddings, c) for c in chunks]

        for f in tqdm(as_completed(futures), total=len(chunks)):
            chunk_text, vectors = f.result()

            if not vectors:
                continue

            if vector_store is None:
                dim = len(vectors[0])
                vector_store = FAISS(
                    embedding_function=OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME),
                    index=faiss.IndexFlatL2(dim),
                    docstore=InMemoryDocstore(),
                    index_to_docstore_id={},
                )

            # 每个 chunk 会被插入多次：每次用不同的“问题向量”作为索引向量
            chunk_text_clean = _replace_tabs(chunk_text)
            pairs = [(chunk_text_clean, vec) for vec in vectors]
            vector_store.add_embeddings(pairs)

    assert vector_store is not None, "向量库初始化失败：没有生成任何向量"
    return vector_store


def encode_pdf(path: Path, *, chunk_size: int, chunk_overlap: int) -> FAISS:
    documents = PyPDFLoader(str(path)).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    splits = splitter.split_documents(documents)

    chunks = [_replace_tabs(d.page_content) for d in splits]
    return prepare_vector_store(chunks)


## 4) 构建 HyPE 向量库

这一步可能会花比较久：因为它会对每个 chunk 调一次 LLM 生成问题，然后再 embedding。

In [5]:
chunks_vector_store = encode_pdf(
    PATH,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks_vector_store

  0%|          | 0/97 [00:00<?, ?it/s]

## 5) 创建 retriever 并快速验证检索结果

这里检索阶段就和普通 RAG 一样了：

- `as_retriever(k=3)`
- `retriever.invoke(query)`

区别只在于：向量库里每个 chunk 有多条向量入口（来自不同“假设问题”）。

In [6]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is the main cause of climate change?"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
context = list(set(context))

show_context(context)

Context 1:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal

Context 2:
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global clima

## 6) （可选）快速评估

沿用原仓库的“快速打分”风格：

- 自动生成一些测试问题
- 用 retriever 拉回上下文
- 让 LLM 对检索质量打分（Relevance / Completeness / Conciseness）

> 注意：这只是快速 sanity check，不等价于严格评测。

In [7]:
evaluate_rag(chunks_query_retriever)

{'questions': ['1. What are the primary greenhouse gases contributing to climate change, and how do they differ in their sources and effects on the atmosphere?',
  '2. Explain the concept of the carbon footprint and discuss how individual lifestyle choices can impact climate change.',
  '3. Describe the role of deforestation in climate change and its effects on biodiversity and carbon sequestration.',
  '4. How do climate change and extreme weather events, such as hurricanes and droughts, affect global food security?',
  '5. Discuss the potential economic impacts of climate change on developing countries compared to developed countries, highlighting specific challenges faced by each.'],
 'results': ['```json\n{\n  "Relevance": 5,\n  "Completeness": 4,\n  "Conciseness": 4\n}\n```',
  '```json\n{\n  "Relevance": 4,\n  "Completeness": 3,\n  "Conciseness": 3\n}\n```',
  '```json\n{\n  "Relevance": 5,\n  "Completeness": 4,\n  "Conciseness": 3\n}\n```',
  '```json\n{\n  "Relevance": 4,\n  "C